# *Pós-graduação em Inteligência Artificial par Negócios*
### *Trabalho Final - Machine Learning*

**Disciplina:** Machine Learning

**Professor:** André Juan Costa Vieira

**Turma:** 

**Nomes dos Integrantes:** 

1- Paulo Vinicius Mota

2-

3-

4- 

## Qualidade de vinhos. 

Você foi contratado como cientista de dados pela famosa vinícola **"Vini Tradizionali di Manduria"** para analisar todos os aspectos dos vinhos produzidos. Diversas questões foram levantadas, como: Compreender os padrões das características que proporcionam boas safras e a qualidade de novos vinhos produzidos, antes que a comunidade mundial **"Vins Spectaculaires"** os deguste, apontar quais são os novos **"blends"** que podem ter continuidade no desenvolvimento, dentre várias outras atividades que visam as boas tomadas de decisões, sempre com o intuito de servir os melhores rótulos, aumentando os lucros e diminuindo os gastos.        

A equipe de enólogos faz estudos frequentes para verificar as características de cada vinho, colocando-os em planilhas. Para melhor compreensão dos dados, descreveram o que significado de cada propriedade.   


##### Descrição 

**0. Color:** Se o vinho é tinto vermelho ou branco

**1. Fixed Acidity:** Qtd.de Ácido não volátil, aquele que não evapora fácil

**2. Volatile Acidity:** Teor de ácido acético que leva a um sabor desagradável de vinagre

**3. Citric Acid:** Um tipo de ácido que age como conservante para aumentar o nível de acidez em pequena quantidade para adicionar aroma e sabor

**4. Residual Sugar:** Qtd. de açúcar restante depois da fermentação, mais de 45g/litro é doce

**5. Chlorides:** Qtd. de sal

**6. Free Sulfur Dioxide:** Componente que impede crescimento microbiano e a oxidação do vinho

**7. Total Sulfur Dioxide:** Qtd. de SO2 (dióxido de enxofre)

**8. Density:** Densidade do vinho,

**9. pH:** Nível de acidez ou potencial hidrogeniônico

**10. Sulphates:** Um adicional que contribui para níveis de SO2 e é 
antimicróbico e antioxidante 

**11. Alcohol:** Qtd. de álcool

**12. Qualidade:** Notas de 3 a 9

# Questões

### Importe todas as bibliotecas necessárias na célula abaixo
##### Organize-as de forma crescente em relação ao tamanho da frase

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

### Tratamento dos Dados


Sir. Pounce, enólogo de longa data, especializado em vinhos italianos, descobriu que estão faltando alguns valores nas planilhas, e que outros dados foram alterados pelo ex-funcionário Smeagle, dispensado por degustar vinhos 'preciosos'.   

**Utilize o dataset 'wines_preprocessing.csv' para fazer as questões abaixo.**

1- Busque os valores faltantes no dataset e trate-os.

2- Busque valores incongruentes no dataset, imprima e os trate. 

3- Valide seus tratamentos com o dataset **'wines.csv'**, demonstrando se foi possível manter as distribuições de forma adequada.

In [ ]:
df_wines_preprocessing = pd.read_csv('../data/wines_pre_processing.csv')
df_valores_faltantes = df_wines_preprocessing.isnull().sum() #1 - buscando valores faltantes 

#df_wines_preprocessing.dtypes

colunas_numericas = df_wines_preprocessing.columns.drop('color') # Pego as colunas numericas e dropo a coluna categorica color 

incongruentes_por_coluna = {}  # dicionário pra guardar os incongruentes de cada coluna para printar dps

for col in colunas_numericas: #2 loop que percore a coluna buscando valores incongruentes, texto no lugar de numero. 
    coluna_original = df_wines_preprocessing[col]
    coluna_convertida = pd.to_numeric(coluna_original, errors='coerce')
    valores_incongruentes = coluna_convertida.isna() & coluna_original.notna()
    incongruentes_por_coluna[col] = coluna_original[valores_incongruentes]  # <- novo: salva o resultado dessa coluna
    #print("Valor Incongruente : ", coluna_original[valores_incongruentes])
    df_wines_preprocessing[col] = coluna_convertida # troca a coluna  pela versão convertida

#df_wines_preprocessing.dtypes
    
df_wines_preprocessing['color'].unique() # pegando os valores de cores validas 
cores_validas = ['white','red'] # Os valores de cores realmente validas foram white e red 
cores_invalidas = ~ df_wines_preprocessing['color'].isin(cores_validas) & df_wines_preprocessing['color'].notna() # atribuo true nas posições onde o valor da color não e um valor válido tbm não é nulo
colunas_continuas = colunas_numericas.drop(['quality', 'alcohol']) #recebe as colunas numericas e dropa quality e alcohol fica pro Carlos tratar com regressão

for col in colunas_continuas:# percorre as colunas e preenche a mediana onde os valores é nulos
    df_wines_preprocessing[col] = df_wines_preprocessing[col].fillna(df_wines_preprocessing[col].median())

df_wines_preprocessing.isna().sum()  # confere quantos faltantes restam depois de tratar as colunas contínuas

cores_incongruentes_valores = df_wines_preprocessing['color'][cores_invalidas].copy()  # guarda o texto original antes de virar NaN
df_wines_preprocessing.loc[cores_invalidas, 'color'] = np.nan  # substitui os valores incongruentes de color por NaN pra poder tratá-los com fillna

df_wines_preprocessing['quality'] = df_wines_preprocessing['quality'].fillna(df_wines_preprocessing['quality'].mode()[0])  # preenche os faltantes de quality com a moda

df_wines_preprocessing['color'] = df_wines_preprocessing['color'].fillna(df_wines_preprocessing['color'].mode()[0])  # preenche os faltantes de color com a moda

df_wines_preprocessing['quality'] = df_wines_preprocessing['quality'].astype(int)  # converte quality pra inteiro 
df_wines_preprocessing.isna().sum()  # confirma que não tem nenhum valor faltante no dataset

In [ ]:
print("1) VALORES FALTANTES: ")
print(df_valores_faltantes)

faltantes_finais = df_wines_preprocessing.isna().sum()
print("\nFaltantes restantes por coluna:")
print(faltantes_finais)
print("\n alcohol foi deixada de propósito com faltantes, para o Carlos tratar com regressão")
print("Faltantes fora de alcohol:", faltantes_finais.drop('alcohol').sum())

In [ ]:
print("2) VALORES INCONGRUENTES: ")
for col, valores in incongruentes_por_coluna.items():
    if len(valores) > 0:
        print(f"\n{col}:")
        print(valores)

print(f"\nValores incongruentes em 'color':")
print(cores_incongruentes_valores)

In [ ]:
df_wines = pd.read_csv("../data/wines.csv")  # carrega o dataset de referência
print("3) VALIDAÇÃO COM WINES.CSV: ")
display(df_wines.describe())
display(df_wines_preprocessing.describe())

O dono da vinícola, Sir. Donald Shelby, tem um filho cursando especialização em ciência de dados, que, ao deparar-se com o dataset, pediu a você que, ao invés de ficar tratando dados com 'essas' técnicas triviais, fizesse um algoritmo de regressão logística para encontrar os valores faltantes na coluna 'Alcohol'. O Sr. Shelby é um homem conhecido como mafioso e considera seu filho um gênio, por isso, você, com fortes receios de sofrer consequências inusitadas por ordem do Don Corleone da atualidade, acatou o "pedido".


4- Desenvolver um algoritmo de regressão logística visando preencher os dados faltantes da coluna "Alcohol". Em seguida, valide os resultados com o dataset **"wines.csv"**, apresentando todas as métricas de classificação estudadas. 

5- Você, ao ver os resultados encontrados, se adiantou e fez um modelo de regressão polinomial para dirimir a questão. Em seguida, validou os resultados com o dataset **"wines.csv"**, utilizando todas as métricas de regressão estudadas. Por fim, escreverá um e-mail explicando o motivo <u>técnico</u> que o levou a não utilizar a regressão logística neste problema, bem como qual a melhor técnica que encontrou para tratar os valores faltantes.

In [ ]:
# 4) regressão logistica para preencher alcohol

# regressão logistica é modelo de classificação: a saida é a probabilidade de cada classe.
# alcohol é um numero (vai de 8 a 14.9, mais de 100 valores diferentes), então pra usar a logistica ele precisa virar categoria.
# alcohol é cortado em 3 faixas por quantil (baixo, medio, alto), cada faixa com quase a mesma quantidade de vinhos.
# o modelo prevê a faixa e o valor faltante recebe a mediana da faixa. a questão 5 mostra o que se perde nesse corte.

df_reg = df_wines_preprocessing.drop(columns=['Unnamed: 0']).copy()  # coluna de indice que veio do csv, não descreve o vinho
df_reg['color'] = (df_reg['color'] == 'red').astype(int)  # sklearn só aceita numero: red vira 1, white vira 0

linhas_sem_alcohol = df_reg['alcohol'].isna()  # 15 linhas ficaram sem alcohol depois do tratamento (4 vazias e 11 com texto)
print("linhas com alcohol faltando:", linhas_sem_alcohol.sum())

X = df_reg.drop(columns=['alcohol'])  # todas as outras colunas entram como feature, inclusive quality
X_conhecido = X[~linhas_sem_alcohol]
X_faltante = X[linhas_sem_alcohol]
y_conhecido = df_reg.loc[~linhas_sem_alcohol, 'alcohol']
alcohol_real = df_wines.loc[linhas_sem_alcohol, 'alcohol']  # gabarito das 15 linhas, vem do wines.csv carregado na questão 3

nomes_faixas = ['baixo', 'medio', 'alto']
y_faixa, limites = pd.qcut(y_conhecido, q=3, labels=nomes_faixas, retbins=True)
print("limites das faixas:", limites.round(2))
print(y_faixa.value_counts())

mediana_por_faixa = y_conhecido.groupby(y_faixa, observed=True).median()  # valor usado no preenchimento quando o modelo escolhe a faixa
print("\nmediana de cada faixa:")
print(mediana_por_faixa)

# correlação de alcohol com as outras colunas, pra ver quem mais ajuda o modelo. density é a maior: alcool é menos denso que agua
print("\ncorrelação de alcohol com as outras colunas:")
print(df_reg[~linhas_sem_alcohol].corr()['alcohol'].drop('alcohol').sort_values().round(2))

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X_conhecido, y_faixa, test_size=0.2, random_state=42, stratify=y_faixa)  # stratify mantem a proporção das faixas em treino e teste

modelo_log = Pipeline([
    ('escala', StandardScaler()),  # as colunas tem escalas bem diferentes (density ~0.99, total sulfur dioxide ~100), a logistica precisa de tudo na mesma escala
    ('logistica', LogisticRegression(max_iter=2000)),
])
modelo_log.fit(X_treino, y_treino)

faixa_prevista = modelo_log.predict(X_teste)
proba_prevista = modelo_log.predict_proba(X_teste)

print("METRICAS DE CLASSIFICAÇÃO (teste, 20% dos vinhos com alcohol conhecido)")
print(f"acuracia : {accuracy_score(y_teste, faixa_prevista):.3f}")
print(f"precisao : {precision_score(y_teste, faixa_prevista, average='macro'):.3f}")
print(f"recall   : {recall_score(y_teste, faixa_prevista, average='macro'):.3f}")
print(f"f1-score : {f1_score(y_teste, faixa_prevista, average='macro'):.3f}")
print(f"roc auc  : {roc_auc_score(y_teste, proba_prevista, multi_class='ovr'):.3f}")  # ovr pq são 3 classes: calcula uma curva ROC por faixa e tira a media
print("\nrelatorio por faixa:")
print(classification_report(y_teste, faixa_prevista))

matriz = confusion_matrix(y_teste, faixa_prevista, labels=nomes_faixas)
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues', xticklabels=nomes_faixas, yticklabels=nomes_faixas)
plt.xlabel('faixa prevista')
plt.ylabel('faixa real')
plt.title('Matriz de confusão - regressão logistica')
plt.show()

In [ ]:
# aplica o modelo nas 15 linhas sem alcohol e compara com o gabarito do wines.csv
faixa_faltante = modelo_log.predict(X_faltante)
proba_faltante = modelo_log.predict_proba(X_faltante)
alcohol_logistica = pd.Series(mediana_por_faixa.loc[faixa_faltante].values, index=X_faltante.index)  # preenchimento: mediana da faixa prevista

faixa_real = pd.cut(alcohol_real, bins=limites, labels=nomes_faixas, include_lowest=True)  # faixa em que o valor verdadeiro cai, com os mesmos cortes

comparacao_log = pd.DataFrame({
    'alcohol_real': alcohol_real,
    'faixa_real': faixa_real,
    'faixa_prevista': faixa_faltante,
    'alcohol_preenchido': alcohol_logistica,
})
comparacao_log['erro'] = (comparacao_log['alcohol_preenchido'] - comparacao_log['alcohol_real']).round(2)
display(comparacao_log)

print("VALIDAÇÃO COM WINES.CSV (15 linhas)")
print(f"acuracia : {accuracy_score(faixa_real, faixa_faltante):.3f}")
print(f"precisao : {precision_score(faixa_real, faixa_faltante, average='macro', zero_division=0):.3f}")
print(f"recall   : {recall_score(faixa_real, faixa_faltante, average='macro', zero_division=0):.3f}")
print(f"f1-score : {f1_score(faixa_real, faixa_faltante, average='macro', zero_division=0):.3f}")
print(f"roc auc  : {roc_auc_score(faixa_real, proba_faltante, multi_class='ovr'):.3f}")
print("\nmatriz de confusão (linha = real, coluna = prevista, ordem baixo/medio/alto):")
print(confusion_matrix(faixa_real, faixa_faltante, labels=nomes_faixas))

# mesmo com a faixa certa, o valor preenchido é a mediana da faixa. em graus de alcool o erro fica assim:
print(f"\nMAE do preenchimento : {mean_absolute_error(alcohol_real, alcohol_logistica):.3f}")
print(f"RMSE do preenchimento: {np.sqrt(mean_squared_error(alcohol_real, alcohol_logistica)):.3f}")

In [ ]:
# 5) regressão polinomial

# aqui alcohol volta a ser previsto como numero. os graus 1, 2 e 3 são comparados com validação cruzada no treino, antes de olhar o teste
X_treino_r, X_teste_r, y_treino_r, y_teste_r = train_test_split(X_conhecido, y_conhecido, test_size=0.2, random_state=42)

def monta_polinomial(grau):
    return Pipeline([
        ('escala', StandardScaler()),
        ('poli', PolynomialFeatures(degree=grau, include_bias=False)),  # cria as colunas ao quadrado, ao cubo e as multiplicações entre colunas
        ('linear', LinearRegression()),
    ])

r2_por_grau = {}
for grau in [1, 2, 3]:
    r2_cv = cross_val_score(monta_polinomial(grau), X_treino_r, y_treino_r, cv=5, scoring='r2')
    r2_por_grau[grau] = r2_cv.mean()
    qtd_termos = monta_polinomial(grau).fit(X_treino_r, y_treino_r)['poli'].n_output_features_
    print(f"grau {grau}: {qtd_termos:>3} termos | R2 medio na validação cruzada = {r2_cv.mean():.3f} (desvio {r2_cv.std():.3f})")

melhor_grau = max(r2_por_grau, key=r2_por_grau.get)
print("\nmelhor grau:", melhor_grau)  # grau 3 cria centenas de colunas a partir de 12, decora o treino e o R2 despenca fora dele

In [ ]:
modelo_poli = monta_polinomial(melhor_grau).fit(X_treino_r, y_treino_r)
alcohol_previsto_teste = modelo_poli.predict(X_teste_r)

def metricas_regressao(y_real, y_previsto):
    return pd.Series({
        'MAE': mean_absolute_error(y_real, y_previsto),
        'MSE': mean_squared_error(y_real, y_previsto),
        'RMSE': np.sqrt(mean_squared_error(y_real, y_previsto)),
        'MAPE (%)': mean_absolute_percentage_error(y_real, y_previsto) * 100,
        'R2': r2_score(y_real, y_previsto),
    })

print(f"METRICAS DE REGRESSÃO (teste, grau {melhor_grau})")
print(metricas_regressao(y_teste_r, alcohol_previsto_teste).round(3))

alcohol_polinomial = pd.Series(modelo_poli.predict(X_faltante), index=X_faltante.index)
print("\nVALIDAÇÃO COM WINES.CSV (15 linhas)")
print(metricas_regressao(alcohol_real, alcohol_polinomial).round(3))

# as tres tecnicas lado a lado: mediana (usada nas outras colunas na questão 1), logistica e polinomial
comparacao_final = pd.DataFrame({
    'alcohol_real': alcohol_real,
    'mediana': y_conhecido.median(),
    'logistica': alcohol_logistica,
    'polinomial': alcohol_polinomial.round(2),
})
display(comparacao_final)

resumo = pd.DataFrame({
    tecnica: metricas_regressao(alcohol_real, comparacao_final[tecnica])
    for tecnica in ['mediana', 'logistica', 'polinomial']
}).T.round(3)
print("erro de cada tecnica nas 15 linhas, contra o wines.csv:")
display(resumo)

fig, eixos = plt.subplots(1, 2, figsize=(13, 4.5))
eixos[0].scatter(y_teste_r, alcohol_previsto_teste, alpha=0.3, s=12)
eixos[0].plot([8, 15], [8, 15], color='red', linewidth=1)  # linha onde previsto = real
eixos[0].set_xlabel('alcohol real')
eixos[0].set_ylabel('alcohol previsto')
eixos[0].set_title(f'Polinomial grau {melhor_grau} no conjunto de teste')
comparacao_final[['alcohol_real', 'logistica', 'polinomial']].plot(kind='bar', ax=eixos[1])
eixos[1].set_ylim(7, 15)
eixos[1].set_xlabel('indice da linha no dataset')
eixos[1].set_ylabel('alcohol')
eixos[1].set_title('As 15 linhas preenchidas')
plt.tight_layout()
plt.show()

# o ponto isolado do grafico é o pior erro do teste: um tinto com chlorides 0.61 e sulphates 2.0, os maiores do dataset inteiro.
# o polinomio eleva esses valores ao quadrado e a previsão cai pra 3 graus. vinho muito fora do padrão do treino, o modelo erra feio
previsto_teste = pd.Series(alcohol_previsto_teste, index=y_teste_r.index)
pior = (previsto_teste - y_teste_r).abs().idxmax()
print(f"\npior erro no teste: linha {pior}, real {y_teste_r[pior]}, previsto {previsto_teste[pior]:.2f}")
display(X_teste_r.loc[[pior]])

# preenchimento final com a polinomial, que teve o menor erro
df_wines_preprocessing.loc[linhas_sem_alcohol, 'alcohol'] = alcohol_polinomial.round(2)
print("faltantes em alcohol depois do preenchimento:", df_wines_preprocessing['alcohol'].isna().sum())

#### E-mail (questão 5)

Para: Sir. Donald Shelby

Assunto: Coluna Alcohol: por que a regressão logística não foi usada no preenchimento

Sr. Shelby,

O preenchimento dos 15 valores que faltavam na coluna Alcohol está concluído. A regressão logística solicitada foi implementada e avaliada, mas a versão final da planilha usa uma regressão polinomial.

A regressão logística é um algoritmo de classificação: ela devolve a probabilidade de uma amostra pertencer a uma categoria. O teor alcoólico é uma medida contínua, vai de 8,0 a 14,9 graus e tem mais de cem valores diferentes na planilha. Para aplicar a logística foi preciso transformar o teor alcoólico em categorias. O Alcohol foi cortado em três faixas de tamanho parecido (baixo, até 9,7; médio, até 11,0; alto, acima disso) e o modelo foi treinado para prever a faixa. Ele acerta a faixa em 82% dos vinhos de teste e a área sob a curva ROC ficou em 0,945, ou seja, como classificador está bom. O problema aparece na hora de preencher: como o modelo só sabe a faixa, o valor que entra é a mediana dela (9,3, 10,4 ou 11,9). Um vinho de 13,1 graus, por exemplo, recebeu 11,9 mesmo com a faixa certa. Nas 15 linhas que faltavam o erro médio foi de 0,51 grau. Transformar número em categoria joga informação fora, e ela não volta na hora de preencher.

Na regressão polinomial o alvo continua sendo o número. Os graus 1, 2 e 3 foram comparados por validação cruzada e o grau 2 foi o melhor. O grau 3 cria 454 colunas a partir de 12, decora os dados de treino e erra muito fora deles. No conjunto de teste o modelo de grau 2 explicou 85% da variação do teor alcoólico (R² de 0,854) com erro médio de 0,31 grau. Nas mesmas 15 linhas o erro médio caiu para 0,40 grau, contra 0,51 da logística e 1,09 se a mediana da coluna fosse repetida, como foi feito nas outras colunas. A variável que mais ajuda o modelo é a densidade: álcool é menos denso que água, então vinho com mais álcool tem densidade menor, e essa relação aparece forte nos dados (correlação de -0,69).

A recomendação é manter a regressão polinomial de grau 2 para preencher os valores faltantes de Alcohol. A planilha tratada já está com esses valores.

Carlos

### Análise Exploratória
**Utilize o dataset 'wines.csv'**

A enóloga Marilyn Monroe, direta do Sir. Pounce, tomou conhecimento de suas habilidades exploratórias e requereu gráficos "chiques, reuscados, enfeitados e nada triviais" que mostrassem, de forma interativa todos os dados e seus respectivos insights. A principal exigência é de que as paletas de cores sejam harmônicas, de modo que possam ser utilizadas em apresentações. Para isso, sugeriu a documentação a seguir:
[Colors Palettes](https://plotly.com/python/builtin-colorscales/)

1- Utilize um countplot para averiguar a quantidade de vinhos por cada avaliação de qualidade. 
Separare entre vinhos tintos e brancos, fazendo um gráfico para cada tipo.

2- Utilize um jointplot para descrever a relação entre álcool e açucar. Utilizar o tipo 'KDE'.

3- Utilize um boxplot para verificar se existe algum vinho que seja considerado um outlier. Utilize **x = 'quality'** e **y='residual sugar'**. Identificando os outliers, crie um novo dataframe, utilize um barplot para contabilizar a quantidade de vinhos tintos
e brancos por qualidade de modo que as barras estejam sobrepostas em relação ao tipo de vinho.   

4- Faça um gráfico de correlação e encontre quais são as 'features' que contém correlações
positivas e negativas fortes  entre si. Em seguida, utilize o scatterplot, colocando no eixo "x" e "y"
cada variável correlata e descreva por escrito o motivo da distribuição e o sentido vetorial estarem apresentados
das respectivas formas.

**ps**: Para este problema, entenda como correlações fortes valores menores que -0.4 e maiores que 0.4.

In [28]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np 

df_wines = pd.read_csv('../data/wines.csv')

paleta_tipo = {'red': '#7B241C', 'white': '#D4AC0D'}  

**Countplot de qualidade, separado por tipo de vinho**

In [18]:
df_tinto = df_wines[df_wines['color'] == 'red']
df_branco = df_wines[df_wines['color'] == 'white']

fig_tinto = px.histogram(
    df_tinto, x='quality', color='quality',
    color_discrete_sequence=px.colors.sequential.Sunsetdark,
    title='Vinhos Tintos - Contagem por Qualidade', text_auto=True
)
fig_tinto.update_layout(template='plotly_white', showlegend=False, xaxis=dict(dtick=1), title_x=0.5)
fig_tinto.show()

fig_branco = px.histogram(
    df_branco, x='quality', color='quality',
    color_discrete_sequence=px.colors.sequential.YlOrBr,
    title='Vinhos Brancos - Contagem por Qualidade', text_auto=True
)
fig_branco.update_layout(template='plotly_white', showlegend=False, xaxis=dict(dtick=1), title_x=0.5)
fig_branco.show()

a distribuição de qualidade é parecida entre tintos e brancos,
concentrada nas notas 5, 6 e 7. Poucos vinhos recebem notas extremas.

**Jointplot (KDE) - Álcool x Açúcar Residual**

In [19]:
fig = make_subplots(
    rows=2, cols=2,
    column_widths=[0.8, 0.2], row_heights=[0.2, 0.8],
    horizontal_spacing=0.02, vertical_spacing=0.02
)

fig.add_trace(
    go.Histogram2dContour(
        x=df_wines['alcohol'], y=df_wines['residual sugar'],
        colorscale='Tealrose', showscale=False,
        contours=dict(coloring='heatmap')
    ), row=2, col=1
)
fig.add_trace(
    go.Histogram(x=df_wines['alcohol'], marker_color='#7b241C', nbinsx=40, showlegend=False),
    row=1,col=1
)
fig.add_trace(
    go.Histogram(y=df_wines['residual sugar'], marker_color='#D4AC0D', nbinsy=40, showlegend=False),
    row=2, col=2
)

fig.update_layout(title_text='Densidade KDE - Álcool x Açúcar Residual', title_x=0.5,
                  template='plotly_white', height=600, width=700)

fig.update_xaxes(title_text='Álcool(%)', row=2, col=1)
fig.update_yaxes(title_text='Açúcar Residual(g/L)', row=2, col=1)
fig.show()

Maior densidade em açúcar residual baixo (< 5 g/L) e álcool entre 9% e 12,5%.
Existe uma cauda de vinhos mais doces (açúcar residual > 20 g/L), tipicamente com teor alcoólico um pouco mais baixo.

**Boxplot de outliers (quality x residual sugar) e barplot sobreposto**

In [20]:
fig = px.box(
    df_wines, x='quality', y='residual sugar', color='quality',
    color_discrete_sequence= px.colors.sequential.Sunsetdark,
    points='outliers', title='Boxplot - Açúcar Residual por Qualidade'
)

fig.update_layout(template='plotly_white', showlegend=False, title_x=0.5)
fig.show()

In [21]:
q1 = df_wines.groupby('quality')['residual sugar'].transform(lambda s: s.quantile(0.25))
q3 = df_wines.groupby('quality')['residual sugar'].transform(lambda s: s.quantile(0.75))
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

eh_outlier = (df_wines['residual sugar'] < limite_inferior) | (df_wines['residual sugar'] > limite_superior)
df_outliers = df_wines[eh_outlier].copy() #novo dataframe com os outliers

print(f'Total de vinhos outliers: {len(df_outliers)}')
df_outliers[['color', 'quality', 'residual sugar']].sort_values('residual sugar', ascending=False).head(10)

Total de vinhos outliers: 147


,color,quality,residual sugar
4391,white,6,65.80
5275,white,6,31.60
5708,white,6,31.60
1982,white,6,26.05
3593,white,6,26.05
1871,white,5,23.50
4165,white,5,22.60
1366,white,5,22.00
3589,white,5,22.00
2152,white,6,20.80


In [25]:
contagem = df_outliers.groupby(['quality', 'color']).size().reset_index(name='contagem')

fig = px.bar(
    contagem, x='quality', y='contagem', color='color',
    color_discrete_map=paleta_tipo, barmode='overlay', opacity=0.75,
    title='Vinhos Outliers(Açúcar Residual) por Qualidade - Tinto x Branco'
)

fig.update_layout(template='plotly_white', title_x=0.5, xaxis=dict(dtick=1))
fig.show()

A maioria dos outliers de açúcar residual é de vinhos brancos, coerente
com a existência de brancos bem mais doces no dataset (raro entre os tintos).

**Correlação entre features**

In [26]:
corr = df_wines.select_dtypes('number').corr()

fg = px.imshow(
    corr, text_auto='.2f', color_continuous_scale='Tealrose', zmin=1, zmax=1,
    title='Mapa de Correlação - Variáveis Físico-Químicas'

)
fig.update_layout(template='plotly_white', title_x=0.5, height=600)
fig.show()



In [29]:
mascara = np.triu(np.ones(corr.shape),k=1).astype(bool)
pares = corr.where(mascara).stack().reset_index()
pares.columns = ['var1', 'var2', 'corr']
fortes = pares[(pares['corr'] > 0.4) | (pares['corr'] < -0.4)].sort_values('corr', ascending=False)
fortes

,var1,var2,corr
66,free sulfur dioxide,total sulfur dioxide,0.720934
43,residual sugar,density,0.552517
42,residual sugar,total sulfur dioxide,0.495482
7,fixed acidity,density,0.458910
131,alcohol,quality,0.444319
41,residual sugar,free sulfur dioxide,0.402871
18,volatile acidity,total sulfur dioxide,-0.414476
94,density,alcohol,-0.686745


**Pares com correlação forte encontrados:**
- "free sulfur dioxide" x "total sulfur dioxide": r = 0.72 (positiva)
- "residual sugar" x "density": r = 0.55 (positiva)
- "residual sugar" x "total sulfur dioxide": r = 0.50 (positiva)
- "fixed acidity" x "density": r = 0.46 (positiva)
- "alcohol" x "quality": r = 0.44 (positiva)
- "residual sugar" x "free sulfur dioxide": r = 0.40 (positiva)
- "volatile acidity' x "total sulfur dioxide": r = -0.41 (negativa)
- "density" x "alcohol": r = -0.69 (negativa)

**scatterplots para a correlação positiva e negativa mais fortes.**

In [31]:
fig = px.scatter(
    df_wines, x='free sulfur dioxide', y='total sulfur dioxide', color='color',
    color_discrete_map=paleta_tipo, opacity=0.5, trendline='ols',
    title='SO2 Livre x SO2 Total (r = 0.72)'
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

Por que a nuvem de pontos sobe (correlação positiva)?
O SO2 total é, a soma do SO2 livre com o SO2 combinado a outras moléculas de vinho. Como o SO2 livre é uma parte do SO2 total, é esperado que, quanto mais SO2 livre um vinho tiver, maior tende a ser SO2 total, por isso a relação é proporcional.

In [32]:
fig = px.scatter(
    df_wines, x='density', y='alcohol', color='color',
    color_discrete_map=paleta_tipo, opacity=0.5, trendline='ols',
    title='Densidade x Álcool (r = -0.69)'
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

Por que a nuvem de pontos desce (correlação negativa)?
Durante a fermentação, o açúcar(com mais densidade que a água) é convertido em álcool(menos densidade).Vinhos com maior teor alcoólico tiveram uma fermentação mais completa,restando menos residuos de açúcar, o que reduz a densidade final do vinho. Por isso a dispersão demonstra uma relação inversa proporcional.

# Modelos Supervisionados 

### Classificação
**Utilize o dataset 'wine_classification.csv'.**

Após alguns meses, o filho do Sir. Donald Shelby, Chuck Norris Shelby, mais conhecido como "El Chavo del Ocho", em decorrência de seu "notório" saber e comportamento extrovertido, ~para ser eufemista~, foi promovido a "*head*" de Machine Learning, vulgo seu chefe. 

Com suas inusitadas e inovadoras ideias, pediu que você criasse três modelos de árvores, um do tipo "random" e dois do tipo "boost", pois havia descoberto que a otimização pelo gradiente descendente era considerada como "*The American Dream*". Não obstante, gostaria de analisar o gráfico de importância das features.

Ademais, requereu que utilizasse o algoritmo SVM, pelo fato do "kernel trick" performar bem em problemas de altas dimensionalidades. Um KNN "cairia bem também, vamos utilizar por mero desencargo de consciência", disse.  

Em seu discurso inflamado, se pronunciava: "Precisamos realizar tais façanhas nunca vistas na história da Inteligência Artificial, desde que as redes neurais foram introduzidas por Walter Pitts e Warren McCulloch em 1943. Vamos predizer tudo que quisermos, independentemente da uva utilizada na produção. Eu transformarei nossa vinícola na melhor do mundo, pois sou detentor do saber". Tudo dito numa reunião contendo 12 pessoas, trabalhadores braçais inclusos. Tal discurso invejou os oráculos delfos e os lembraram de Sócrates em seu julgamento, antes de morrer. 

Após tamanhas proclamações, apontou em sua direção e disse: **"VOCÊ, É..., VOCÊ MESMO**, irás fazer todo o processo por conta própria, e eu direi se o que fazes está correto! Não utilizarás Auto-ML, pois eu, ~professor~, quero ter certeza de que entende seu labor e suas nuâncias". 

Você, cansado e entediado de tantas lorotas, se retirou da reunião com "dores" na região abdominal, porém ainda recebeu um e-mail lhe instruindo a comparar os resultados de todas as implementações, escolher o melhor modelo e utilizar métodos de otimização de hiperparâmetro.  

Em suma?

1- Crie um pipeline que contenha ao menos 05 tipos diferentes de algoritmos de classificação. 

2- Crie um DataFrame que contenha todos os resultados de todos os algoritmos utilizados, inclusive a métrica ROC AUC.

3- Comparar os resultados, escolher o melhor modelo e otimizar os parâmetros. Ao fim, faça um gráfico da ROC AUC.


Após todos seus esforços, o amado chefe lhe pediu para utilizar um algoritmo de classificação que ele ouviu falar, criado pelo matemático inglês Thomas Bayes. Cabe a você, mais uma vez aplicar o algoritmo e apresentar os resultado. Em seguida, faça uma breve explicação do principal problema desse método para solucionar problemas complexos.  

Dr. Anton Ego marcou uma data para comparecer na vinícula e degustar seus melhores rótulos. Nascido na França e o enólogo mais famoso do mundo, Anton era temido pelas suas análises minuciosas e certeiras. As críticas eram tão serveras que tão severas que 80% das vinículas eram fechadas pela falta de aceitação do mercado. Apenas =~ 19.99% sobreviviam sem danos consideráveis e somente $0.1x10^{-15}$% se tornavam uma lenda.

Chuck tomava leite da papoula para suportar tamanha disruptura emocional. Sir.Donald, tomado pela a ansiedade, estava com seus pruridos mentais em Nárnia até que sua esposa, Srta.Audrey Hepburn assumiu a liderança do projeto com a serenidade de um bebê.

Primeiramente ordenou que todos os vinhos que já vinham há algum tempo em processo de envelhecimento em barricas de carvalho fossem engarrafados e que amostras de todos fossem coletadas para análise. 

Sua maior preocupação é que somente sejam servidos os vinhos de nota oito ou nove, pois ambos são de mesmíssima qualidade, ficando a avaliação a critério da subjetividade palatal do degustador. Em **<u>hipótese nenhuma</u>** um vinho que não tenha tais notas pode ser servido.

De todas as novas garrafas, serão servidas somente três que você autorizar. O Dr. Ego só toma vinho tinto!


Sabendo que você já tinha um modelo validado para solucionar este tipo de problema, pediu que o usasse com a base **'desafio.csv''**. Ao fim, crie uma célula e copie os 3 vinhos que escolheu para registrar sua resposta. 
